<a href="https://colab.research.google.com/github/maabmusa7/BinX_Tech_Internship_Project_GROUP5/blob/MohammadAbuHamed_LLM-preparation/ai-ml/PreprocessingForData2Sets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preprocessing Summary

Two datasets were prepared for fine-tuning the English Tutor LLM.

### 1. DailyDialog

DailyDialog is used to teach the model how to continue natural English conversations.

Each conversation originally contains turns such as:

`Person A: ...`  
`Person B: ...`

During preprocessing:

- `Person A` is converted to the `user` role.
- `Person B` is converted to the `assistant` role.
- The speaker labels are removed from the text.
- A `system` message is added to define the model as an English conversation tutor.

The final format becomes:

`system → user → assistant`

This dataset mainly teaches the model how to maintain conversation flow and respond naturally.

### 2. W&I + LOCNESS

W&I + LOCNESS is used to teach the model how to recognize and correct English learner mistakes.

Each sample contains:

- The learner's original text.
- The learner's CEFR level.
- A set of grammatical edits.

During preprocessing:

- The grammatical edits are applied to reconstruct the corrected version of the learner's text.
- The edits are applied from the end of the text to the beginning to preserve the original character positions.
- The learner's CEFR level is included in the `system` message.
- The original learner text becomes the `user` message.
- The corrected text becomes the `assistant` response.

The final format also becomes:

`system → user → assistant`

This dataset mainly teaches the model grammar correction and how to work with English learners at different proficiency levels.

### Final Result

After preprocessing, both datasets use the same chat structure:

`system → user → assistant`

However, each dataset teaches a different skill:

- **DailyDialog** → Natural conversation.
- **W&I + LOCNESS** → Grammar correction and learner-level awareness.

Using the same chat structure will make it easier to combine the datasets later for fine-tuning the English Tutor LLM.

# Preprocess for 2 data sets

## DailyDialog Preprocessing

The DailyDialog conversations must be converted into a structured chat format before being combined with the other training datasets.

Each utterance will be assigned either a `user` or `assistant` role while preserving the original multi-turn conversation.

The processed dataset will use the same `messages` structure that will later be used across all training datasets.

In [2]:
from datasets import load_dataset
daily_dialog = load_dataset("elricwan/dailydialog")
df = daily_dialog["train"].to_pandas()
df.head()

README.md:   0%|          | 0.00/348 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.06MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/13118 [00:00<?, ? examples/s]

,conversation
0,"[Person A: The kitchen stinks ., Person B: I'l..."
1,"[Person A: So Dick , how about getting some co..."
2,[Person A: Are things still going badly with y...
3,"[Person A: Would you mind waiting a while ?, P..."
4,[Person A: Are you going to the annual party ?...


In [3]:
def convert_daily_dialog(conversation):
  messages=[]
  system_text="You are an English conversation tutor. Continue the conversation naturally"
  messages.append({
        'role':'system',
        'content':system_text
    })
  for sentence in conversation:
    person,text=sentence.split(':',1)
    role = "uesr" if person=='Person A' else "assistant"
    messages.append({
        'role':role,
        'content':text.strip()
    })

  return messages


In [4]:
df["messages"] = df["conversation"].apply(convert_daily_dialog)

In [5]:
df['messages'].loc[0]

[{'role': 'system',
  'content': 'You are an English conversation tutor. Continue the conversation naturally'},
 {'role': 'uesr', 'content': 'The kitchen stinks .'},
 {'role': 'assistant', 'content': "I'll throw out the garbage ."}]

In [6]:
daily_dialog_ready = df[["messages"]].copy()

In [7]:
daily_dialog_ready["source"] = "DailyDialog"

In [8]:
daily_dialog_ready.head()

,messages,source
0,"[{'role': 'system', 'content': 'You are an Eng...",DailyDialog
1,"[{'role': 'system', 'content': 'You are an Eng...",DailyDialog
2,"[{'role': 'system', 'content': 'You are an Eng...",DailyDialog
3,"[{'role': 'system', 'content': 'You are an Eng...",DailyDialog
4,"[{'role': 'system', 'content': 'You are an Eng...",DailyDialog


## W&I + LOCNESS Preprocessing

The W&I + LOCNESS dataset contains English learner texts together with grammatical correction information and CEFR proficiency levels.

Before combining this dataset with the other training datasets, we need to transform each learner sample into a structured chat format suitable for fine-tuning the English Tutor LLM.

In [9]:
from datasets import load_dataset
grammar_correct1=load_dataset("martinsr/wi_locness")
df2=grammar_correct1["train"].to_pandas()
df2.head()

README.md:   0%|          | 0.00/5.31k [00:00<?, ?B/s]

wi_locness_train.parquet: reconstructing file:   0%|          |  0.00B / 3.59MB            

wi_locness_train.parquet: downloading bytes:           |  0.00B            

wi_locness_dev.parquet: reconstructing file:   0%|          |  0.00B /  469kB            

wi_locness_dev.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/350 [00:00<?, ? examples/s]

,text,edits,cefr,id
0,My town is a medium size city with eighty thou...,"{'start': [13, 77, 104, 126, 134, 256, 306, 37...",A2.i,1-140178
1,Everyone has his own plans. People want to be ...,"{'start': [28, 56, 106, 118, 124, 131, 152, 17...",A2.i,1-181175
2,Now days each family has more then 1 car for e...,"{'start': [0, 30, 50, 75, 79, 86, 131, 178, 20...",A2.ii,1-59315
3,"Furthermore, the biggest group of positive fee...","{'start': [34, 184, 210, 234, 249, 274, 287, 3...",A1.i,1-232774
4,MORE THAN KIP FIT!\n\nDo you know mountain bik...,"{'start': [10, 31, 100, 127, 130, 181, 197, 27...",A2.ii,1-352674


### Inspect Dataset Columns

We first inspect the available columns to understand the structure of the dataset and identify the fields that will be used during preprocessing.

In [10]:
df2.columns

Index(['text', 'edits', 'cefr', 'id'], dtype='object')

### Inspect a Sample Record

A sample record is displayed to understand how the learner text, CEFR level, and grammatical correction information are stored.

In [11]:
sample = grammar_correct1["train"][0]
for key, value in sample.items():
    print(f"\n--- {key} ---")
    print(value)


--- text ---
My town is a medium size city with eighty thousand inhabitants. It has a high density population because its small territory. Despite of it is an industrial city, there are many shops and department stores.  I recommend visiting the artificial lake in the certer of the city which is surrounded by a park. Pasteries are very common and most of them offer the special dessert from the city. There are a comercial zone along the widest street of the city where you can find all kind of establishments: banks, bars, chemists, cinemas, pet shops, restaurants, fast food restaurants, groceries, travel agencies, supermarkets and others. Most of the shops have sales and offers at least three months of the year: January, June and August. The quality of the products and services are quite good, because there are a huge competition, however I suggest you taking care about some fakes or cheats.

--- edits ---
{'start': [13, 77, 104, 126, 134, 256, 306, 375, 396, 402, 476, 484, 579, 671, 77

### Reconstructing the Corrected Text

The `correct_text()` function applies all grammatical edits to the learner's original text and returns the fully corrected version.

The function works as follows:

1. `zip(edits["start"], edits["end"], edits["text"])` combines the start position, end position, and replacement text for each correction.

2. `list(...)` converts the combined edits into a list.

3. `reversed(...)` processes the edits from the end of the text to the beginning. This prevents earlier corrections from changing the character positions of later corrections.

4. `for start, end, new_text in ...` goes through each correction one by one.

5. If `new_text` is `None`, it means that the incorrect part should be deleted, so it is replaced with an empty string `""`.

6. `text = text[:start] + new_text + text[end:]` replaces the incorrect part with the corrected text by combining:
   - the text before the mistake,
   - the corrected text,
   - the text after the mistake.

### Simple Example

Original text:

`I go school`

Correction information:

`start = 2`  
`end = 4`  
`new_text = "went to"`

The function combines:

`"I " + "went to" + " school"`

Final corrected text:

`I went to school`

After all edits are applied, the function returns the complete corrected text.

In [27]:
def correct_text(text, edits):
    for start, end, new_text in reversed(
        list(zip(edits["start"], edits["end"], edits["text"]))):
        if new_text is None:
            continue
        text = text[:start] + new_text + text[end:]
    return text

In [28]:
sample = grammar_correct1["train"][0]
corrected_text = correct_text(
    sample["text"],
    sample["edits"]
)
print("Original Text:\n")
print(sample["text"])
print("\nCorrected Text:\n")
print(corrected_text)

Original Text:

My town is a medium size city with eighty thousand inhabitants. It has a high density population because its small territory. Despite of it is an industrial city, there are many shops and department stores.  I recommend visiting the artificial lake in the certer of the city which is surrounded by a park. Pasteries are very common and most of them offer the special dessert from the city. There are a comercial zone along the widest street of the city where you can find all kind of establishments: banks, bars, chemists, cinemas, pet shops, restaurants, fast food restaurants, groceries, travel agencies, supermarkets and others. Most of the shops have sales and offers at least three months of the year: January, June and August. The quality of the products and services are quite good, because there are a huge competition, however I suggest you taking care about some fakes or cheats.

Corrected Text:

My town is a medium-sized city with eighty thousand inhabitants. It has a hi

### Converting the Corrected Samples to Chat Format

After reconstructing the corrected text, each sample is converted into a chat format.

The chat contains:

- A `system` message that defines the model as an English tutor and provides the student's CEFR level.
- A `user` message containing the learner's original text.
- An `assistant` message containing the corrected text.

In [29]:
def convert_grammar_sample(row):
    corrected = correct_text(
        row["text"],
        row["edits"]
    )
    messages = [
        {
            "role": "system",
            "content": f"You are an English tutor. The student's CEFR level is {row['cefr']}. Correct the learner's English."
        },
        {
            "role": "user",
            "content": row["text"]
        },
        {
            "role": "assistant",
            "content": corrected
        }
    ]
    return messages

### Testing the Chat Conversion

The conversion function is tested on one sample to verify that the original learner text, CEFR level, and corrected text are correctly transformed into the final chat format.

In [30]:
sample = grammar_correct1["train"][0]
messages = convert_grammar_sample(sample)
for message in messages:
    print(message["role"].upper())
    print(message["content"])
    print()

SYSTEM
You are an English tutor. The student's CEFR level is A2.i. Correct the learner's English.

USER
My town is a medium size city with eighty thousand inhabitants. It has a high density population because its small territory. Despite of it is an industrial city, there are many shops and department stores.  I recommend visiting the artificial lake in the certer of the city which is surrounded by a park. Pasteries are very common and most of them offer the special dessert from the city. There are a comercial zone along the widest street of the city where you can find all kind of establishments: banks, bars, chemists, cinemas, pet shops, restaurants, fast food restaurants, groceries, travel agencies, supermarkets and others. Most of the shops have sales and offers at least three months of the year: January, June and August. The quality of the products and services are quite good, because there are a huge competition, however I suggest you taking care about some fakes or cheats.

ASSIS

### Apply the Chat Conversion to the Full Dataset

After verifying that the conversion works correctly on a sample, the same preprocessing function is applied to all records in the training dataset.

Each learner sample is converted into a structured `system`, `user`, and `assistant` conversation.

In [32]:
df2["messages"] = df2.apply(convert_grammar_sample,axis=1)

In [33]:
df2[["text", "cefr", "messages"]].head()

,text,cefr,messages
0,My town is a medium size city with eighty thou...,A2.i,"[{'role': 'system', 'content': 'You are an Eng..."
1,Everyone has his own plans. People want to be ...,A2.i,"[{'role': 'system', 'content': 'You are an Eng..."
2,Now days each family has more then 1 car for e...,A2.ii,"[{'role': 'system', 'content': 'You are an Eng..."
3,"Furthermore, the biggest group of positive fee...",A1.i,"[{'role': 'system', 'content': 'You are an Eng..."
4,MORE THAN KIP FIT!\n\nDo you know mountain bik...,A2.ii,"[{'role': 'system', 'content': 'You are an Eng..."


In [34]:
for message in df2["messages"].iloc[0]:
    print(message["role"].upper())
    print(message["content"])
    print()

SYSTEM
You are an English tutor. The student's CEFR level is A2.i. Correct the learner's English.

USER
My town is a medium size city with eighty thousand inhabitants. It has a high density population because its small territory. Despite of it is an industrial city, there are many shops and department stores.  I recommend visiting the artificial lake in the certer of the city which is surrounded by a park. Pasteries are very common and most of them offer the special dessert from the city. There are a comercial zone along the widest street of the city where you can find all kind of establishments: banks, bars, chemists, cinemas, pet shops, restaurants, fast food restaurants, groceries, travel agencies, supermarkets and others. Most of the shops have sales and offers at least three months of the year: January, June and August. The quality of the products and services are quite good, because there are a huge competition, however I suggest you taking care about some fakes or cheats.

ASSIS

In [35]:
wi_locness_ready = df[["messages"]].copy()

In [36]:
wi_locness_ready["source"] = "W&I + LOCNESS"